# **RI HuggingFace File Scanning Demo**

> ▶️ **Try this in Colab!** Run the [RI Adversarial File Scanning Walkthrough in Google Colab](https://colab.research.google.com/github/RobustIntelligence/docs/blob/main/notebooks/demo_notebooks/RI_Adversarial_File_Scanning_Walkthrough.ipynb). 

You are the AI Risk Officer at a Consumer Social Company. The NLP team has been tasked with implementing a text classification model to predict the top-level "sentiment" of posts on the app. These predictions will later be consumed by multiple models throughout the company, such as recommendation, lead prediction, and the core advertisement models. You want to verify your models are sufficiently robust to adversaries seeking to exploit model vulnerabilities and boost content that your user base does not actually like.

In this Notebook Walkthrough, we will review our core product of **AI Stress Testing** of NLP models in an *adversarial setting*. RIME AI Stress Testing allows you to test any text classification model on any dataset. In this way, you will be able to quantify your model's vulnerability to attacks and noisy data.

Your team's NLP models are fine-tuned from state-of-the-art transformer models found on [Hugging Face's Model Hub 🤗](https://huggingface.co/models). In particular, you have chosen to fine-tune a [DistilBERT](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) on data similar to the [Stanford Sentiment Treebank](https://huggingface.co/datasets/sst2) dataset for a lightweight yet performant model. 

For more information on how to connect with a [Hugging Face Model](https://readthedocs.com/cas/login?service=https%3A%2F%2Frobust-intelligence-inc-rime.readthedocs-hosted.com%2Fen%2Flatest%2Ffor_data_scientists%2Fhow_to_guides%2Fintegrations%2Fhuggingface.html%3Fnext%3Dhttps%253A%252F%252Frobust-intelligence-inc-rime.readthedocs-hosted.com%252Fen%252Flatest%252Ffor_data_scientists%252Fhow_to_guides%252Fintegrations%252Fhuggingface.html%253Fhighlight%253Dhuggingface#huggingface-classification-model) or 
[Hugging Face Dataset](https://readthedocs.com/cas/login?service=https%3A%2F%2Frobust-intelligence-inc-rime.readthedocs-hosted.com%2Fen%2Flatest%2Ffor_data_scientists%2Freference%2Fconfiguration%2Fnlp%2Fdata_source.html%3Fnext%3Dhttps%253A%252F%252Frobust-intelligence-inc-rime.readthedocs-hosted.com%252Fen%252Flatest%252Ffor_data_scientists%252Freference%252Fconfiguration%252Fnlp%252Fdata_source.html%253Fhighlight%253Dhuggingface#huggingface-dataset), check out the linked documentation.

To begin, please specify your RIME cluster's URL and personal access token.

## 1. **Install Dependencies, Import Libraries, and Download Data**

In [ ]:
# Installing the dependencies
%pip install rime-sdk
%pip install python-dotenv

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from rime_sdk import Client
import os
from dotenv import load_dotenv, find_dotenv

## 2. **Establish the RI Client**

To get started, provide the API credentials and the base domain/address of the RIME service. You can generate and copy an API token from the API Access Tokens Page under Workspace settings. For the domian/address of the RIME service, contact your admin. 

In [ ]:
#Load environment variables
load_dotenv(find_dotenv())
API_TOKEN = os.environ.get('API_TOKEN')
CLUSTER_URL = os.environ.get('CLUSTER_URL')
AGENT_ID = os.environ.get('AGENT_ID')
WORKSPACE_ID = os.environ.get('WORKSPACE_ID')

In [ ]:
client = Client(CLUSTER_URL, API_TOKEN)

## 3. **Create a New Project**

Below, create a project to store this and other future adversarial robustness stress test run results.

In [ ]:
description = (
    "Evaluate the robustness of text classification models"
    " against adversarial attacks. Demonstration uses the"
    " DAIR-Emotion dataset (https://huggingface.co/datasets/dair-ai/emotion)"
    " and a fine-tuned version of the DistilBERT model"
    " (https://huggingface.co/esuriddick/distilbert-base-uncased-finetuned-emotion)."
)
project = client.create_project(
    name="HuggingFace File Scanning", 
    description=description,
    model_task="MODEL_TASK_MULTICLASS_CLASSIFICATION"
)

## 4. **Uploading the Model and Datasets**



##### 4.1. Registering two HuggingFace models

In [ ]:
#Benign model
distilbert_model_id = project.register_model(
    name = f"distilbert_{datetime.now()}",
    model_config={
        "hugging_face": {
            "model_uri": "esuriddick/distilbert-base-uncased-finetuned-emotion"
        }
    },
    agent_id=AGENT_ID
)

# Malicious model
malicious_bert_model_id = project.register_model(
    name = f"bert_{datetime.now()}",
    model_config={
        "hugging_face": {
            "model_uri": "drhyrum/bert-tiny-torch-picklebomb"
        }
    },
    agent_id=AGENT_ID
)

##### 4.2. Uploading Reference and Evaluation datasets from HuggingFace

In [ ]:
ref_dataset_id = project.register_dataset(
    name = f'train_emotion_dataset_{datetime.now()}',
    data_config = {
        "connection_info": {
            "hugging_face": {
                "dataset_uri": "dair-ai/emotion",
                "split_name": "train",
            },
        },
        "data_params": {
            "label_col": "label",
            "text_features": ["text"],
            "class_names": ["sadness", "joy","love","anger","fear","surprise"],
            "sample": True,
            "nrows": 100
        },
    }
)

eval_dataset_id = project.register_dataset(
    name = f'validation_emotion_dataset_{datetime.now()}',
    data_config = {
        "connection_info": {
            "hugging_face": {
                "dataset_uri": "dair-ai/emotion",
                "split_name": "validation",
            },
        },
        "data_params": {
            "label_col": "label",
            "text_features": ["text"],
            "class_names": ["sadness", "joy","love","anger","fear","surprise"],
            "sample": True,
            "nrows": 100
        },
    }
)

## 5. **Running a File Scan on the Registered Models**

In [ ]:
#Benign model
file_scan_job_distilbert = client.start_file_scan(
    model_id = distilbert_model_id, 
    project_id = project.project_id, 
    agent_id = AGENT_ID
)

#Malicious model
file_scan_job_malicious_bert = client.start_file_scan(
    model_id = malicious_bert_model_id, 
    project_id = project.project_id, 
    agent_id = AGENT_ID
)

file_scan_job_malicious_bert.get_status(wait_until_finish=True), file_scan_job_distilbert.get_status(wait_until_finish=True)

## 6. **File Scan Results**

In [ ]:
file_scan_result_malicious_bert = client.get_file_scan_result(file_scan_id = file_scan_job_malicious_bert.job_id)
file_scan_result_distilbert = client.get_file_scan_result(file_scan_id = file_scan_job_distilbert.job_id)

print(f"BERT (malicious) file scanning result: {file_scan_result_malicious_bert['severity']}")
print(f"DistilBERT file scanning result: {file_scan_result_distilbert['severity']}") 

## 7. **Running a Stress Test**
This test is on the benign distilbert model

In [ ]:
stress_test_config = {
    "run_name": "DistilBERT Adversarial Robustness",
    "data_info": {
        "ref_dataset_id": ref_dataset_id,
        "eval_dataset_id": eval_dataset_id,
    },
    "model_id": distilbert_model_id,
    "categories": [
        "TEST_CATEGORY_TYPE_ABNORMAL_INPUTS",
        "TEST_CATEGORY_TYPE_ADVERSARIAL",
        "TEST_CATEGORY_TYPE_BIAS_AND_FAIRNESS",
        "TEST_CATEGORY_TYPE_DATA_CLEANLINESS",
        "TEST_CATEGORY_TYPE_DRIFT",
        "TEST_CATEGORY_TYPE_DATA_POISONING_DETECTION",
        "TEST_CATEGORY_TYPE_MODEL_PERFORMANCE",
        "TEST_CATEGORY_TYPE_SUBSET_PERFORMANCE",
        "TEST_CATEGORY_TYPE_SUBSET_PERFORMANCE_DEGRADATION",
        "TEST_CATEGORY_TYPE_TRANSFORMATIONS",
    ],
    "run_time_info": {
        "resource_request": {
            "ram_request_megabytes": "28000",
        },
        "random_seed" : "0"
    }
}
stress_job = client.start_stress_test(
    stress_test_config, project.project_id, agent_id=AGENT_ID
)
stress_job.get_status(verbose=True, wait_until_finish=True)

## 8. **Analyzing and Querying Results**

Now that the test run is complete, we can check out the results in the RIME web interface.

In [ ]:
test_run = stress_job.get_test_run()
results_df = test_run.get_result_df()
results_df.head()

In [ ]:
# Get a link to the stress test
print("https://"+ test_run.get_link())